# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: CTR / Engagement Opportunity Scoring.**

I am choosing this lane because the target signal is directly observable
in the current data rather than requiring a proxy label built from a
future window, which keeps out the labeling-design risk that Refresh
Scoring and Growth/Recovery both carry. Comparing CTR only within the
same position tier controls for the largest known driver of CTR by
design, and the output is a diagnostic tied to a specific, testable
editorial action (title or snippet review) rather than a general
priority score whose underlying cause is harder to isolate.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.chdir('./../../')
assert os.path.exists("./data/raw/content_refresh_anonymized.csv")
print("Lane: CTR / Engagement Opportunity Scoring")

Lane: CTR / Engagement Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision this improves: out of all visible pages, which ones are ranking
well (good average position) but still getting fewer clicks than other
pages at that same position, and should be reviewed for title, meta
description, or snippet changes first.

Who acts on it: a content editor or SEO strategist, with limited review
capacity, deciding which page titles/metadata to rewrite this week.

Cost of a wrong call:
- False positive (page flagged as CTR-weak but the low CTR is normal for
  that query type, e.g. informational intent): wastes editor time
  rewriting something that was not actually broken.
- False negative (a page that is genuinely under-capturing clicks is
  missed): a page keeps ranking well but silently loses traffic it could
  have earned, and no one investigates it.
Both costs are wasted or lost opportunity, since review capacity is the
limited resource, not the ranking itself.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Total pages:", df.shape[0])
print("Pages with avg_position between 1 and 20:", ((df["avg_position"] > 0) & (df["avg_position"] <= 20)).sum())

Total pages: 30000
Pages with avg_position between 1 and 20: 20256


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = df[df["impressions_90d"] >= 100].copy()

ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (impressions_90d >= 100):")
print(ctr_by_tier.round(4))

visible["tier_avg_ctr"] = visible.groupby("position_tier")["ctr"].transform("mean")
candidates = visible[(visible["position_tier"] == "striking") & (visible["ctr"] < visible["tier_avg_ctr"])]
print("\nStriking-position pages below their tier's average CTR:", candidates.shape[0])

Mean CTR by position tier (impressions_90d >= 100):
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554
Name: ctr, dtype: float64

Striking-position pages below their tier's average CTR: 3883


These numbers make the lane worth pursuing: CTR clearly depends on
position tier (striking-position pages average 0.412 CTR versus 0.061
for page_3_5), so comparing CTR across tiers would be misleading -
comparing within a tier is the correct approach. Over 2,100 pages sit
in the best position tier but still underperform their own tier's
average CTR, meaning there is a large, well-ranked group of pages
genuinely leaving clicks on the table.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work can say:
- Observed: this page's CTR is below the average CTR of other pages at
  the same position tier, given a minimum impression volume.
- Directional: pages with these gaps are more likely to be under-serving
  their ranking position, based on this dataset's pattern.
- Decision-support: this ranked list is a starting point for review, to
  be checked by a human before any title or metadata change is made.

What this work will never say:
- That a low CTR is caused by a bad title or snippet specifically -
  the data cannot confirm the cause, only the gap.
- That rewriting a title will increase CTR - that requires an
  experiment (e.g. before/after or A/B test), not this data alone.
- That any result explains or predicts Google's ranking or snippet
  algorithm.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Claims: observed, directional, decision-support only")
print("Never claim: cause of low CTR, guaranteed improvement, Google algorithm behavior")

Claims: observed, directional, decision-support only
Never claim: cause of low CTR, guaranteed improvement, Google algorithm behavior


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.